# Dasar Machine Learning

## Bagian 1e: Pengujian Hipotesis


## Tujuan

Setelah mempelajari bagian ini, kamu dapat:

- Membedakan hipotesis nol dan hipotesis alternatif.
- Menyusun sendiri pasangan hipotesis untuk diuji.


In [ ]:
# Imports
import warnings

import matplotlib.pyplot as plt
import numpy as np
from scipy import stats
from scipy.stats import binom

warnings.filterwarnings("ignore")


## Hipotesis nol

Misalkan seseorang mengaku punya kemampuan meramal hasil lempar koin. Kita ingin menguji klaim itu. Percobaannya sederhana: ia menebak 100 lemparan, dan 57 di antaranya benar.

Apakah angka itu cukup untuk membuktikan kemampuannya?

### Menentukan hipotesis nol dan alternatif

**Hipotesis nol:** orang itu tidak istimewa; peluang tebakan benarnya tetap 0.5.

**Hipotesis alternatif:** peluang tebakan benarnya lebih besar dari 0.5.

Besaran yang diukur dari percobaan, atau **statistik uji**, adalah jumlah tebakan benar dari 100 lemparan.

Bila hipotesis nol benar, statistik uji mengikuti distribusi binomial dengan `n = 100` dan `p = 0.5`. Pertanyaannya: seberapa besar peluang memperoleh 57 tebakan benar atau lebih, jika hipotesis nol benar?

Sebelum percobaan dijalankan, ambang keputusan ditetapkan lebih dulu. Ambang 5% adalah pilihan yang lazim: bila data se-ekstrem hasil pengamatan hanya muncul kurang dari 5% kesempatan saat hipotesis nol benar, hipotesis nol ditolak.


In [ ]:
# Peluang memperoleh 57 tebakan benar atau lebih bila p = 0.5
p_value = 1 - binom.cdf(56, 100, 0.5)
print(f"{p_value * 100:.1f}%")


Peluangnya sekitar 9.7%, lebih besar dari 5%. Hipotesis nol tidak ditolak, sehingga belum ada bukti bahwa orang tersebut benar-benar bisa meramal.

Sebaliknya, ambang 95% bisa dihitung lebih dulu untuk mengetahui berapa tebakan benar yang dianggap meyakinkan.

### Latihan

Berapa jumlah tebakan benar yang membuat seseorang layak dianggap punya kemampuan meramal?


In [ ]:
# Nilai kritis pada tingkat keyakinan 95%
cutoff = binom.ppf(0.95, 100, 0.5) + 1
print(cutoff)


Diperlukan 59 tebakan benar atau lebih agar seseorang dianggap punya kemampuan prediksi pada tingkat keyakinan 95%.

## Klaim yang lebih spesifik

Bagaimana bila klaimnya berubah: "Saya tidak sempurna, tetapi bisa menebak benar 60% dari waktu." Saat hasilnya 57, klaim tersebut tetap bisa ditolak dengan cara yang sama. Namun orang itu mungkin berdalih hasilnya kurang beruntung hari itu.

Pada kasus ini tidak ada satu hipotesis nol tunggal, melainkan dua hipotesis yang bersaing:

- Hipotesis A: tidak ada kemampuan prediksi (peluang 50%).
- Hipotesis B: kemampuan prediksi 60%.

Keduanya dapat digambarkan sebagai kurva normal, satu berpusat di 50 dan satu di 60.


In [ ]:
def plot_competing_curves(trials, rate_a, rate_b, x_limits):
    """Gambarkan dua kurva normal yang mewakili hipotesis A dan B."""
    grid = np.linspace(x_limits[0], x_limits[1], 400)
    for rate, label in ((rate_a, "A"), (rate_b, "B")):
        mean = trials * rate
        std_dev = np.sqrt(trials * rate * (1 - rate))
        plt.plot(grid, stats.norm.pdf(grid, mean, std_dev), label=f"Hipotesis {label}")
    plt.xlim(*x_limits)
    plt.xlabel("Jumlah tebakan benar")
    plt.ylabel("Kepadatan peluang")
    plt.legend()
    plt.show()

plot_competing_curves(100, 0.5, 0.6, (30, 80))


Hasil 57 tebakan benar masih masuk akal pada kedua kurva.


In [ ]:
# Peluang 57 tebakan atau lebih pada masing-masing hipotesis
print(1 - binom.cdf(57, 100, 0.5))
print(binom.cdf(57, 100, 0.6))


Data yang ada belum cukup untuk mencapai keyakinan 95%. Ada beberapa cara mengatasinya.

### 1) Menurunkan tingkat keyakinan

Karena klaimnya 50 dan 60, titik tengah 55 bisa dipakai sebagai ambang: tebakan di atas 55 dianggap punya kemampuan prediksi. Konsekuensinya, tingkat keyakinan ikut turun.


In [ ]:
print(1 - binom.cdf(55, 100, 0.5))
print(binom.cdf(54, 100, 0.6))


Pada tingkat keyakinan sekitar 87%, hasil 57 tebakan sudah cukup untuk menyimpulkan adanya kemampuan prediksi. Cara ini kurang ideal karena ambangnya dipilih berdasarkan angka tengah, bukan interval keyakinan.

### 2) Membuat ambang tidak seimbang

Kedua hipotesis tidak harus diberi porsi kesalahan yang sama. Titik potong bisa digeser ke mana saja. Pada ambang 55, peluang menolak kemampuan yang sebenarnya ada sekitar 13%, dan peluang menerima kemampuan yang sebenarnya tidak ada juga sekitar 13%. Menggeser ambang akan menurunkan satu jenis kesalahan tetapi menaikkan yang lain.

Misalnya ambang 95% untuk koin yang adil dipertahankan:


In [ ]:
# Kebalikan dari fungsi cdf di atas
binom.ppf(0.95, 100, 0.5)


Ambangnya menjadi 58. Kedua jenis kesalahannya adalah:


In [ ]:
print(1 - binom.cdf(58, 100, 0.5))
print(binom.cdf(58, 100, 0.6))


Ada 5% peluang menyimpulkan seseorang punya kemampuan padahal tidak. Namun ada 38% peluang menolak seseorang yang sebenarnya punya kemampuan. Bergantung pada kebutuhan, ketidakseimbangan ini bisa jadi justru yang diinginkan. Kedua jenis kesalahan ini disebut kesalahan Tipe I dan Tipe II.

### 3) Memperbesar ukuran sampel

Pilihan terbaik adalah menambah jumlah percobaan. Bayangkan percobaannya dilakukan 1000 kali, bukan 100 kali.


In [ ]:
plot_competing_curves(1000, 0.5, 0.6, (450, 650))


Dengan 1000 percobaan, kedua kurva normal hampir tidak lagi bertumpuk.


In [ ]:
print(binom.ppf(0.95, 1000, 0.5))
print(binom.ppf(0.05, 1000, 0.6))


### Latihan

Berapa besar kedua kesalahan pada kondisi baru ini?


In [ ]:
print(1 - binom.cdf(550, 1000, 0.5))
print(binom.cdf(550, 1000, 0.6))


Dengan ambang 550, kedua kesalahan turun di bawah 0.1%. Menambah ukuran sampel membuat keputusan menjadi jauh lebih meyakinkan tanpa perlu menggeser tingkat keyakinan.
